# Pooled regions 1d: sites from the middle backward (third CPU session)

A copy of Section 1.1–1.4 of `Pooled Regions 1 - Sites and Fits for Every MPRA Region.ipynb` that builds the first half of the TF list, from the middle back toward the start. Four sessions can run at once: main (start → forward), 1b (end → backward), 1d (middle → backward), 1e (middle → forward); a TF already in `sites/` is skipped, so they meet. **CPU runtime.** Run it top to bottom; 1.5 is not here.

In [ ]:
!pip install -q pymemesuite pyfaidx requests
from google.colab import drive
drive.mount('/content/drive')

## 1.1 Settings and every MPRA region

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil
import numpy as np, pandas as pd

ACCESSORY = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/accessary_files"
MPRA_PATH = f"{ACCESSORY}/mpra_sequences_with_embeddings_nt.pkl"   # a .csv with the same columns also works
REGIONS = None                # None = every region in the MPRA file; or a list such as [0, 1, 8, 20]
CELLS = ["HepG2", "K562"]     # ChIP-seq and DNase from these cells, sites pooled
N_PER_CLASS = 2000            # at most this many sites per class after matching
MATCH_VARS = ["pwm", "gc", "cpg_oe"]
N_MATCH_BINS = 5              # quantile bins per matching variable (5 x 5 x 5 joint bins)
UNBOUND_POOL = 10             # unbound candidates collected: UNBOUND_POOL x N_PER_CLASS, before matching
BOUND_TOP = 0.25              # bound: peaks in the top quarter by signalValue within each experiment
UNBOUND_MIN_DIST = 1000       # unbound: at least this far from every peak of the TF, relaxed peaks included

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
SHARE = [f"{WORK_DIR}/{d}/sites" for d in ("fcav_pgc_hk", "fcav_pgc_hk_r0", "fcav_pgc_hk_r20")]   # sites built before
for d in ("", "/sites", "/regions"):
    os.makedirs(POOL + d, exist_ok=True)
json.dump({"regions": REGIONS, "cells": CELLS, "n_per_class": N_PER_CLASS, "match_vars": MATCH_VARS,
           "n_match_bins": N_MATCH_BINS, "unbound_pool": UNBOUND_POOL, "bound_top": BOUND_TOP,
           "unbound_min_dist": UNBOUND_MIN_DIST}, open(f"{POOL}/settings.json", "w"), indent=1)
TOK, MOFF, WLEN = C["tok"], C["motif_offset"], C["window_len"]
pd.set_option("display.width", 250)
_RC = str.maketrans("ACGT", "TGCA")


def revcomp(s):
    return s.translate(_RC)[::-1]


_m = pd.read_csv(MPRA_PATH, low_memory=False) if MPRA_PATH.endswith(".csv") else pd.read_pickle(MPRA_PATH)
_m = _m.drop(columns=[c for c in ("wt_embedding", "mut_embedding", "sequence_mut") if c in _m.columns])
_m["region_id"] = _m["region_id"].astype(int)
if REGIONS is not None:
    _m = _m[_m["region_id"].isin(REGIONS)]
SEQ, CELL, HINT, SNV0 = {}, {}, {}, {}
for rid, g in _m.groupby("region_id"):
    s = g["sequence"].dropna().astype(str).str.upper().unique()
    assert len(s) == 1, f"region {rid}: {len(s)} different WT sequences"
    SEQ[rid] = s[0]
    CELL[rid] = ",".join(sorted(g["cellline"].astype(str).unique())) if "cellline" in g.columns else "?"
    HINT[rid] = str(g["chr"].iloc[0]) if "chr" in g.columns else None
    v = g[(g["alt"] != "-") & g["log2FC"].notna()].copy()
    v["local"] = v["pos"].astype(int) - v["start"].astype(int)
    v["ref"], v["alt"] = v["ref"].astype(str).str.upper(), v["alt"].astype(str).str.upper()
    assert all(0 <= p < len(SEQ[rid]) and SEQ[rid][p] == r for p, r in zip(v["local"], v["ref"])), f"region {rid}: ref mismatch"
    SNV0[rid] = v.groupby(["local", "ref", "alt"]).agg(log2FC=("log2FC", "mean"), n_meas=("log2FC", "size")).reset_index()
del _m
print(pd.DataFrame({"cell": CELL, "bp": {r: len(s) for r, s in SEQ.items()},
                    "SNVs": {r: len(s) for r, s in SNV0.items()}}).to_string())

In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
from pyfaidx import Fasta

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
BASE_IDX = {b: i for i, b in enumerate("ACGT")}


def freq(name):
    f = np.asarray(MOTIF[name].frequencies, float)
    return f / f.sum(1, keepdims=True)


def log_odds(name, strand):
    f = freq(name) if strand == "+" else freq(name)[::-1, [3, 2, 1, 0]]
    return np.log2(np.clip(f, 1e-9, 1.0) / 0.25)


def site_score(seq, name, strand):
    lo = log_odds(name, strand)
    return float(sum(lo[j, BASE_IDX[b]] for j, b in enumerate(seq)))


def scan_many(name, seqs, batch=500):
    out = [[] for _ in seqs]
    for i0 in range(0, len(seqs), batch):
        chunk = [Sequence(s, name=str(i0 + k).encode()) for k, s in enumerate(seqs[i0:i0 + batch])]
        for h in FIMO_SCAN.score_motif(MOTIF[name], chunk, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            out[int(h.source.name.decode())].append((a - 1, b, h.strand, h.pvalue))
    return out


LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROMS = [c for c in [f"chr{i}" for i in range(1, 23)] + ["chrX"] if c in GENOME]
CHROM_LEN = {c: len(GENOME[c]) for c in CHROMS}


def locate(seq, hint):
    # a 40 bp anchor from each end must land at the right distance; <= 2% mismatches allowed
    n, A = len(seq), 40
    order = ([f"chr{hint}".replace("chrchr", "chr")] if hint else []) + CHROMS
    for c in dict.fromkeys(order):
        if c not in GENOME:
            continue
        chrom = GENOME[c][:].seq
        for strand, q in (("+", seq), ("-", revcomp(seq))):
            i = chrom.find(q[:A])
            while i >= 0:
                if chrom[i + n - A:i + n] == q[-A:]:
                    mm = sum(a != b for a, b in zip(chrom[i:i + n], q))
                    if mm <= 0.02 * n:
                        return c, i, strand, mm
                i = chrom.find(q[:A], i + 1)
    return None


# every region in genome orientation: GSEQ, its SNVs (local position, ref/alt on the + strand), its interval
LOC = {}
for r in sorted(SEQ):
    loc = locate(SEQ[r], HINT[r])
    if not loc:
        print(f"region {r}: not found in hg38 -- skipped")
        continue
    chrom, g0, strand, mm = loc
    snv = SNV0[r].copy()
    if strand == "-":
        gseq = revcomp(SEQ[r])
        snv["local"] = len(SEQ[r]) - 1 - snv["local"]
        snv["ref"], snv["alt"] = snv["ref"].str.translate(_RC), snv["alt"].str.translate(_RC)
    else:
        gseq = SEQ[r]
    snv = snv.sort_values(["local", "alt"]).reset_index(drop=True)
    assert all(gseq[p] == b for p, b in zip(snv["local"], snv["ref"]))
    LOC[r] = dict(chrom=chrom, g0=g0, strand=strand, mm=mm, gseq=gseq, snv=snv)
    print(f"region {r:2d} ({CELL[r]}): {chrom}:{g0}-{g0 + len(gseq)} ({strand}), {mm} mismatches to hg38, {len(snv)} SNVs")
REGION_IVS = {r: (L["chrom"], L["g0"], L["g0"] + len(L["gseq"])) for r, L in LOC.items()}

## 1.2 Every JASPAR hit with SNVs inside, and ΔPWM, per region

Same scan as the per-region notebooks (FIMO, `fimo_p` from the config, both strands); a hit is kept when at least one
measured SNV lies inside it. Hit ids are per region, in FIMO-p order. Saved to `fcav_pool/regions/r<k>/`.

In [ ]:
RL = sorted(LOC)
_raw = {r: [] for r in RL}
for name in MOTIF:                                  # one FIMO call per motif over all regions (p-values do not
    for r, hs in zip(RL, scan_many(name, [LOC[r]["gseq"] for r in RL])):   # depend on the other sequences)
        _raw[r] += [(name, a, b, st, p) for a, b, st, p in hs]
HITS, PAIR = {}, {}
for r in RL:
    gseq, snv = LOC[r]["gseq"], LOC[r]["snv"]
    loc_ = snv["local"].values
    H = pd.DataFrame([{"motif": nm, "start": a, "end": b, "strand": st, "p": float(p), "pwm": site_score(gseq[a:b], nm, st)}
                      for nm, a, b, st, p in _raw[r] if ((loc_ >= a) & (loc_ < b)).any()],
                     columns=["motif", "start", "end", "strand", "p", "pwm"])
    H = H.sort_values(["p", "start", "motif"], kind="mergesort").reset_index(drop=True)
    H["hit_id"] = np.arange(len(H))
    rows = []
    for h in H.itertuples():
        lo = log_odds(h.motif, h.strand)
        for i in np.flatnonzero((loc_ >= h.start) & (loc_ < h.end)):
            k = int(loc_[i]) - h.start
            rows.append((i, h.hit_id, lo[k, BASE_IDX[snv.at[i, "alt"]]] - lo[k, BASE_IDX[snv.at[i, "ref"]]]))
    HITS[r], PAIR[r] = H, pd.DataFrame(rows, columns=["snv", "hit_id", "dpwm"])
    d = f"{POOL}/regions/r{r}"
    os.makedirs(d, exist_ok=True)
    snv.to_csv(f"{d}/snvs.csv", index=False)
    H.to_csv(f"{d}/hits.csv", index=False)
    PAIR[r].to_csv(f"{d}/snv_hit_dpwm.csv", index=False)
    print(f"region {r:2d}: {len(H)} hits of {H['motif'].nunique()} motifs contain SNVs; "
          f"{PAIR[r]['snv'].nunique()} of {len(snv)} SNVs inside at least one hit")
ALL_MOTIFS = sorted(set().union(*(set(H["motif"]) for H in HITS.values())))
print(f"\n{len(ALL_MOTIFS)} motifs with a hit in at least one region")

## 1.3 ENCODE, and the sites already built

A motif gets a direction when an ENCODE TF ChIP-seq target in HepG2 or K562 has its name (ignoring case); heterodimer
matrices (`A::B`) are skipped. Sites built earlier for regions 8 / 0 / 20 are copied, and every matched pair whose
bound or unbound window overlaps **any** MPRA region is dropped (the pair, so the matching stays intact).

In [ ]:
import requests

ENC, HDR = "https://www.encodeproject.org", {"accept": "application/json"}
ENC_DIR = f"{WORK_DIR}/encode"                      # shared download cache with the other notebooks
os.makedirs(ENC_DIR, exist_ok=True)
PEAK_COLS = ["chrom", "start", "end", "name", "score", "strand", "signal", "p", "q", "summit"]
PRIORITY = ["conservative IDR thresholded peaks", "optimal IDR thresholded peaks",
            "IDR thresholded peaks", "replicated peaks", "pseudoreplicated peaks"]


def enc_search(**params):
    r = requests.get(f"{ENC}/search/", params={**params, "format": "json", "limit": "all"},
                     headers=HDR, timeout=600)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return r.json()["@graph"]


def narrowpeak_files(assay_title, cell, target=None):
    q = {"type": "Experiment", "assay_title": assay_title, "status": "released", "biosample_ontology.term_name": cell}
    if target:
        q["target.label"] = target
    rows = []
    for e in enc_search(**q):
        for f in enc_search(type="File", dataset=e["@id"], file_format="bed", assembly="GRCh38", status="released"):
            if str(f.get("file_type", "")).startswith("bed narrowPeak"):
                rows.append({"experiment": e["accession"], "accession": f["accession"],
                             "output_type": f.get("output_type", ""),
                             "preferred_default": bool(f.get("preferred_default", False)), "href": f["href"]})
    return pd.DataFrame(rows)


def fetch_bed(row):
    path = f"{ENC_DIR}/{row['accession']}.bed.gz"
    if not os.path.exists(path):
        with requests.get(ENC + row["href"], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.rename(path + ".part", path)
    df = pd.read_csv(path, sep="\t", header=None, compression="gzip").iloc[:, :10]
    df.columns = PEAK_COLS[:df.shape[1]]
    return df


def cell_targets(cell):
    exps = enc_search(type="Experiment", assay_title="TF ChIP-seq", status="released",
                      **{"biosample_ontology.term_name": cell}, field="target.label")
    return {e["target"]["label"].upper(): e["target"]["label"] for e in exps if "target" in e}


def pick_per_experiment(files):
    keep = []
    for _, g in files.groupby("experiment"):
        if g["preferred_default"].any():
            keep.append(g[g["preferred_default"]].iloc[0])
        else:
            rank = g["output_type"].map(lambda t: PRIORITY.index(t) if t in PRIORITY else 99)
            keep.append(g.loc[rank.idxmin()])
    return pd.DataFrame(keep)


def strict_peaks(files):
    # bound: top BOUND_TOP by signal within each experiment; exclusion: every peak file of the TF
    bound = []
    for _, r in pick_per_experiment(files).iterrows():
        d = fetch_bed(r)
        bound.append(d[d["signal"] >= d["signal"].quantile(1 - BOUND_TOP)])
    excl = pd.concat([fetch_bed(r) for _, r in files.iterrows()], ignore_index=True)
    return pd.concat(bound, ignore_index=True), excl

In [ ]:
def overlaps_region(chrom, g_start):
    # the 600 bp window of a site starting at g_start overlaps some MPRA region
    ws = g_start - MOFF
    return any(chrom == c and ws < e and ws + WLEN > s for c, s, e in REGION_IVS.values())


# ---- reuse: the first earlier copy that was built, minus every pair overlapping any region
n_cp, n_drop = 0, 0
for m in ALL_MOTIFS:
    dst = f"{POOL}/sites/{m}.json"
    if "::" in m or os.path.exists(dst):
        continue
    cands = [f"{d}/{m}.json" for d in SHARE if os.path.exists(f"{d}/{m}.json")]
    if not cands:
        continue
    Js = [json.load(open(p)) for p in cands]
    J = next((j for j in Js if j["status"] == "built"), Js[0])
    rec = J["records"]
    if rec:
        nb = sum(r["cls"] == 1 for r in rec)
        assert len(rec) == 2 * nb and all(r["cls"] == 1 for r in rec[:nb])    # bound[i] is paired with unbound[i]
        keep = [i for i in range(nb) if not (overlaps_region(rec[i]["chrom"], rec[i]["g_start"])
                                              or overlaps_region(rec[nb + i]["chrom"], rec[nb + i]["g_start"]))]
        n_drop += nb - len(keep)
        J["records"] = [rec[i] for i in keep] + [rec[nb + i] for i in keep]
        if len(keep) < 50:
            J["status"], J["records"] = "too few sites after dropping MPRA-region overlaps", []
    json.dump(J, open(dst, "w"))
    n_cp += 1
print(f"reused {n_cp} TFs' sites from regions 8/0/20; {n_drop} pairs dropped for overlapping an MPRA region")

LABEL = {c: cell_targets(c) for c in CELLS}
DN_FILES = {}
for c in CELLS:
    f = narrowpeak_files("DNase-seq", c)
    DN_FILES[c] = f[f["preferred_default"]] if f["preferred_default"].any() else f
    assert len(DN_FILES[c]), f"no DNase-seq in {c}"
SOURCE = {}                                         # motif -> {cell: ChIP files}, or the reason it has none
for m in ALL_MOTIFS:
    if "::" in m:
        SOURCE[m] = "dimer matrix"
    elif os.path.exists(f"{POOL}/sites/{m}.json"):
        SOURCE[m] = "cached"
    else:
        fb = {c: narrowpeak_files("TF ChIP-seq", c, LABEL[c][m.upper()]) for c in CELLS if m.upper() in LABEL[c]}
        fb = {c: f for c, f in fb.items() if len(f)}
        SOURCE[m] = fb if fb else f"no ChIP-seq in {' or '.join(CELLS)}"
show = pd.Series({m: (v if isinstance(v, str) else "to build: " + " + ".join(v)) for m, v in SOURCE.items()})
print("  ".join(f"{c}: {len(LABEL[c])} ChIP-seq targets" for c in CELLS))
print(show.value_counts().to_string())

## 1.4 Sites for the new TFs

As in the per-region notebooks. Bound: the best hit (p < `fimo_p`) within `peak_half` of a strict peak's summit, its
centre ≤ `summit_dist` from the summit. Unbound: hits in that cell's DNase peaks ≥ `UNBOUND_MIN_DIST` from every peak
of the TF in every cell with data. Windows of 600 bp with the motif at bp 300, spaced, **none overlapping any MPRA
region**. Matched on PWM, GC and CpG o/e; at most `N_PER_CLASS` pairs. Resumable: a TF already in `sites/` is skipped.

**Four CPU sessions at once:** a TF already in `sites/` is skipped just before it is built. This copy builds
the first half of the TF list, from the middle back toward the start; the main notebook builds from the start, 1b from the end. Run 1.5 in the main notebook only,
after all four are done.

In [ ]:
class Intervals:
    def __init__(self, df):
        self.d = {}
        for c, g in df.groupby("chrom"):
            o = np.argsort(g["start"].values)
            s, e = g["start"].values[o], g["end"].values[o]
            new = np.r_[True, s[1:] > np.maximum.accumulate(e)[:-1]]
            idx = np.flatnonzero(new)
            self.d[c] = (s[idx], np.maximum.reduceat(e, idx))

    def dist(self, chrom, pos):
        if chrom not in self.d:
            return np.inf
        st, en = self.d[chrom]
        i = np.searchsorted(st, pos, side="right") - 1
        d = np.inf
        if i >= 0:
            d = 0 if pos < en[i] else pos - en[i] + 1
        if i + 1 < len(st):
            d = min(d, st[i + 1] - pos)
        return d


def gc_frac(s):
    return (s.count("G") + s.count("C")) / len(s)


def cpg_oe(s):
    return s.count("CG") * len(s) / max(s.count("C") * s.count("G"), 1)


def site_record(m, cls, chrom, g_start, g_end, strand, p, signal=np.nan):
    ws = g_start - MOFF
    if ws < 0 or ws + WLEN > CHROM_LEN[chrom]:
        return None
    if overlaps_region(chrom, g_start):              # no window may overlap any MPRA region
        return None
    win = GENOME[chrom][ws:ws + WLEN].seq
    if "N" in win:
        return None
    return {"cls": cls, "chrom": chrom, "g_start": int(g_start), "strand": strand, "p": float(p), "signal": float(signal),
            "pwm": site_score(win[MOFF:MOFF + g_end - g_start], m, strand), "window": win}


def spaced(recs):
    kept, occ = [], set()
    for r in sorted(recs, key=lambda r: r["p"]):
        b = (r["chrom"], r["g_start"] // WLEN)
        if any((b[0], b[1] + k) in occ for k in (-1, 0, 1)):
            continue
        kept.append(r)
        occ.add(b)
    return kept


def match_pwm(bound, unbound, rng):
    # joint quantile bins of MATCH_VARS over the pooled sites; one bound to one unbound within each bin
    rec = bound + unbound
    if not bound or not unbound:
        return [], []
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    P = pd.DataFrame({v: [r[v] for r in rec] for v in MATCH_VARS})
    P["cls"] = [1] * len(bound) + [0] * len(unbound)
    keys = [pd.qcut(P[v].rank(method="first"), N_MATCH_BINS, labels=False) for v in MATCH_VARS]
    pairs = []
    for _, g in P.groupby(keys):
        b = rng.permutation(np.array(g.index[g["cls"] == 1]))
        u = rng.permutation(np.array(g.index[g["cls"] == 0]))
        k = min(len(b), len(u))
        pairs += list(zip(b[:k], u[:k]))
    pairs = [pairs[i] for i in rng.permutation(len(pairs))[:N_PER_CLASS]]
    return [rec[i] for i, _ in pairs], [rec[j] for _, j in pairs]


def spans(m):
    w = len(freq(m))
    return w, MOFF // TOK, -(-(MOFF + w) // TOK)


def build(m, files_by_cell):
    t0 = time.time()
    w = len(freq(m))
    peaks = {c: strict_peaks(f) for c, f in files_by_cell.items()}
    excl = Intervals(pd.concat([e for _, e in peaks.values()], ignore_index=True))   # every peak, every cell
    bound, unbound, n_pk = [], [], {}
    for c, (bpk, _) in peaks.items():
        pk = bpk[bpk["chrom"].isin(CHROMS)].copy()
        pk["abs_summit"] = np.where(pk["summit"] >= 0, pk["start"] + pk["summit"], (pk["start"] + pk["end"]) // 2)
        pk = pk.sort_values("signal", ascending=False).drop_duplicates(["chrom", "abs_summit"])
        pk = pk[(pk["abs_summit"] >= C["peak_half"]) & (pk["abs_summit"] + C["peak_half"] <= pk["chrom"].map(CHROM_LEN))]
        n_pk[c] = len(pk)
        chroms, summits, sig = pk["chrom"].tolist(), pk["abs_summit"].astype(int).tolist(), pk["signal"].tolist()
        seqs = [GENOME[ch][s - C["peak_half"]:s + C["peak_half"]].seq for ch, s in zip(chroms, summits)]
        for ch, s, sg, hits in zip(chroms, summits, sig, scan_many(m, seqs)):
            if not hits:
                continue
            a, b, st, p = min(hits, key=lambda h: h[3])
            if abs((a + b) / 2 - C["peak_half"]) > C["summit_dist"] or b - a != w:
                continue
            r = site_record(m, 1, ch, s - C["peak_half"] + a, s - C["peak_half"] + b, st, p, sg)
            if r:
                r["cell"] = c
                bound.append(r)
        want, got = UNBOUND_POOL * N_PER_CLASS // len(peaks), 0
        for i0 in range(0, len(DHS[c]), 2000):
            if got >= want:
                break
            chunk = DHS[c][i0:i0 + 2000]
            for (ch, s, e), hits in zip(chunk, scan_many(m, [GENOME[ch][s:e].seq for ch, s, e in chunk])):
                for a, b, st, p in hits:
                    if b - a != w or excl.dist(ch, s + (a + b) // 2) < UNBOUND_MIN_DIST:
                        continue
                    r = site_record(m, 0, ch, s + a, s + b, st, p)
                    if r:
                        r["cell"] = c
                        unbound.append(r)
                        got += 1
    bound, unbound = spaced(bound), spaced(unbound)
    rng = np.random.default_rng(C["seed"])
    nb, nu = len(bound), len(unbound)
    bound, unbound = match_pwm(bound, unbound, rng)
    rec = bound + unbound
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    w, K0, K1 = spans(m)
    ok = min(len(bound), len(unbound)) >= 50
    json.dump({"tf": m, "status": "built" if ok else f"too few sites (bound {nb}, unbound {nu})", "width": w,
               "K0": K0, "K1": K1, "pool_bound": nb, "pool_unbound": nu, "records": rec if ok else []},
              open(f"{POOL}/sites/{m}.json", "w"))
    by_cell = "  ".join(f"{c} {sum(r['cell'] == c for r in bound)}/{sum(r['cell'] == c for r in unbound)}" for c in CELLS)
    print(f"{m:10s} strict peaks {'+'.join(str(n_pk.get(c, 0)) for c in CELLS):>13s}  pool bound {nb:5d} unbound {nu:5d}"
          f" -> matched {len(bound):5d} per class (bound/unbound by cell: {by_cell})"
          f"{'' if ok else '  TOO FEW'}  [{time.time() - t0:.0f}s]", flush=True)


todo = [m for m, v in SOURCE.items() if not isinstance(v, str)]   # the TFs nobody has built yet, in list order
todo = todo[:len(todo) // 2][::-1]     # first half, from the middle back toward the start
print(f"{len(CHROMS)} chromosomes; {len(todo)} TFs to build" + " (first half, middle -> start)")
if todo:
    DHS = {}
    for c in CELLS:
        _d = pd.concat([fetch_bed(r) for _, r in DN_FILES[c].iterrows()], ignore_index=True)
        _d = _d[_d["chrom"].isin(CHROMS)].sample(frac=1, random_state=C["seed"])
        DHS[c] = list(zip(_d["chrom"], _d["start"].astype(int), _d["end"].astype(int)))
        print(f"{c} DNase peaks: {len(DHS[c]):,}")
    del _d
t_all = time.time()
for k, m in enumerate(todo):
    if os.path.exists(f"{POOL}/sites/{m}.json"):     # built in an earlier, interrupted session
        continue
    print(f"[{k + 1}/{len(todo)}] ", end="")
    build(m, SOURCE[m])
print(f"done [{(time.time() - t_all) / 60:.1f} min]")

## End

Done when this half is finished. Run **1.5** in the main notebook (`Pooled Regions 1`) after all four sessions print `done`, not here.

In [ ]:
from google.colab import runtime
runtime.unassign()